# Speculative Decoding

A refresher on **speculative decoding** — how to make autoregressive LLM inference
faster *without* changing the output distribution, by letting a small "draft" model
guess ahead and a large "target" model verify.

**Domain:** LLM Inference, Training & Optimization  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

Autoregressive decoding is **memory-bandwidth bound**, not compute bound. To emit one
token, a large model must stream all of its weights (and the KV cache) through the
accelerator — and it does this *once per token*, sequentially. The matrix math is cheap;
the data movement is the bottleneck. A modern GPU is therefore mostly idle during
generation, waiting on memory.

**Speculative decoding** exploits the slack: a cheap **draft** model proposes a short
run of `γ` candidate tokens, and the expensive **target** model verifies them **all in a
single forward pass** (one pass scores `γ+1` positions just as cheaply as it would score
one, because the pass is bandwidth-bound). A clever acceptance rule keeps the prefix the
target "agrees" with and resamples at the first disagreement.

The headline property: the accepted tokens are distributed **exactly** as if you had
sampled from the target model token-by-token. It is **lossless** — same distribution,
fewer expensive forward passes. Typical speedups are **2–3×** with no quality change.

**Reach for it when:** you serve a large model, latency matters, and you have (or can
train/distill) a much smaller model that agrees with it often. **Skip it when:** the draft
rarely agrees (gibberish prefixes, very different tokenizers), you're already throughput-bound
with huge batches (the GPU is busy — no idle slack to reclaim), or the engineering cost of
hosting two models outweighs the win.

## 2. Mental Model

Think of an **autocomplete typist with an editor looking over their shoulder.**

- The **draft model** is a fast junior typist who bangs out the next few words (`γ` of them).
- The **target model** is the senior editor who reads the whole proposed phrase **in one glance**
  and, for each word, decides "yes, I'd have written that too" or "no — stop here, *this* is the
  right word."
- Every word the editor accepts was free: the junior produced it and the editor confirmed it in a
  single read. The first rejected word is replaced by the editor's own choice, and the junior
  starts again from there.

A round produces **between 1 and γ+1 tokens** for the price of **one** target forward pass.
If the junior is good, most words survive and you fly. If the junior is hopeless, you fall back
to ~1 token per pass — same speed as plain decoding, plus a little overhead.

```
prefix ──▶ DRAFT proposes:  t1  t2  t3  t4        (γ = 4, cheap)
                             │   │   │   │
prefix ──▶ TARGET verifies all 4 in ONE pass ─────┐
                             ✓   ✓   ✗             │  accept t1,t2; reject t3
                                     └─ resample ──┘  → emit t3' from corrected dist
result this round:  t1  t2  t3'   (3 tokens, 1 target pass)
```

## 3. Key Concepts

- **Target model** `p` — the big model whose distribution you actually want to sample from.
- **Draft / proposal model** `q` — a small, fast model that guesses the next `γ` tokens. Often a
  distilled or smaller sibling sharing the **same tokenizer** (this matters — token IDs must line up).
- **γ (gamma) / draft length** — how many tokens to speculate per round. Bigger γ = more potential
  reward per target pass, but more wasted draft work when rejected. Sweet spot is usually 3–7.
- **Acceptance criterion** — for each drafted token `x` (drawn from `q`), accept with probability
  `min(1, p(x)/q(x))`. If accepted, keep it; if rejected, **resample** from the residual
  distribution `norm(max(0, p − q))`. This exact rule is what makes the scheme **lossless**.
- **Acceptance rate α** — the expected fraction of drafted tokens accepted. It is the single number
  that drives the speedup; it depends entirely on how well `q` mimics `p`.
- **Free bonus token** — if all `γ` draft tokens are accepted, the same target pass already gives
  you logits for one *extra* token, so a fully-accepted round yields `γ+1` tokens.
- **Verification pass** — the single target forward over `prefix + γ drafts`, scoring all positions
  at once thanks to the causal mask.
- **Variants** — *self-speculative* (the model drafts with some of its own layers skipped),
  *Medusa* (extra decoding heads predict multiple future tokens), *EAGLE* (feature-level
  autoregression), *n-gram / prompt-lookup* (draft by copying from the prompt — zero extra model).

## 4. Setup

Pure-Python prerequisites (the worked examples below need only **NumPy**):

```bash
%pip install numpy
```

To run the *optional* real-model example in §5.3 you also need Transformers + PyTorch and a
network connection to pull two small checkpoints (gpt2 / distilgpt2, ~500 MB total):

```bash
%pip install "transformers>=4.40" torch
```

That cell is **gated behind an environment-variable check**, so the notebook executes
top-to-bottom on CPU with no downloads unless you opt in.

In [1]:
# Environment probe — what's available in THIS kernel (no downloads, no GPU needed).
import importlib.util
import sys

import numpy as np


def have(mod: str) -> str:
    return "installed" if importlib.util.find_spec(mod) else "not installed"


print(f"python        : {sys.version.split()[0]}")
print(f"numpy         : {np.__version__}")
for m in ("torch", "transformers"):
    print(f"{m:<14}: {have(m)}")

print("\nWorked examples 1 & 2 are pure-NumPy and run regardless of the above.")

python        : 3.13.7
numpy         : 2.5.0
torch         : installed
transformers  : installed

Worked examples 1 & 2 are pure-NumPy and run regardless of the above.


## 5. Worked Examples

### 5.1 The acceptance rule is *lossless* — verify it empirically

The whole trick rests on one claim: draw `x ~ q`, accept it with probability `min(1, p(x)/q(x))`,
and on rejection resample from `norm(max(0, p − q))` — and the token you end up emitting is
distributed **exactly** as `p`. Let's prove it to ourselves with a Monte-Carlo simulation over a
tiny 6-token "vocabulary." If it's truly lossless, the empirical histogram of emitted tokens must
match `p` — *not* `q`, even though `q` did most of the proposing.

In [2]:
rng = np.random.default_rng(0)
V = 6  # tiny vocabulary

# Target distribution p (what we WANT) and a deliberately-different draft q.
p = np.array([0.05, 0.30, 0.10, 0.25, 0.20, 0.10])
q = np.array([0.20, 0.20, 0.20, 0.10, 0.15, 0.15])
assert np.isclose(p.sum(), 1) and np.isclose(q.sum(), 1)

# Residual distribution used on rejection: norm(max(0, p - q)).
residual = np.maximum(p - q, 0.0)
residual = residual / residual.sum()


def speculative_step(p, q, residual):
    """One token via the speculative accept/reject rule. Returns the emitted token id."""
    x = rng.choice(V, p=q)                       # draft proposes x ~ q
    if rng.random() < min(1.0, p[x] / q[x]):     # accept w.p. min(1, p/q)
        return x
    return rng.choice(V, p=residual)             # else resample from the residual


N = 200_000
emitted = np.array([speculative_step(p, q, residual) for _ in range(N)])
empirical = np.bincount(emitted, minlength=V) / N

print("token :   p (target)   empirical   q (draft)")
for t in range(V):
    print(f"  {t}   :     {p[t]:.3f}       {empirical[t]:.3f}      {q[t]:.3f}")
print(f"\nmax |empirical - p| = {np.abs(empirical - p).max():.4f}  (≈0 ⇒ lossless)")
print(f"max |empirical - q| = {np.abs(empirical - q).max():.4f}  (large ⇒ NOT just sampling q)")

token :   p (target)   empirical   q (draft)
  0   :     0.050       0.050      0.200
  1   :     0.300       0.300      0.200
  2   :     0.100       0.101      0.200
  3   :     0.250       0.250      0.100
  4   :     0.200       0.200      0.150
  5   :     0.100       0.100      0.150

max |empirical - p| = 0.0005  (≈0 ⇒ lossless)
max |empirical - q| = 0.1501  (large ⇒ NOT just sampling q)


The emitted distribution tracks **`p`**, not `q` — confirming the scheme reproduces the target
model exactly, even though the fast draft model generated most of the proposals. That is the
guarantee that lets you bolt speculative decoding onto a production model with zero quality risk.

### 5.2 How much faster? The speedup is governed by acceptance rate α and draft length γ

A round costs **one** target forward pass and yields a random number of tokens. With per-token
acceptance probability `α` and draft length `γ`, the expected number of tokens accepted per round is
a geometric-series sum, and a fully-accepted round earns the free bonus token:

$$\mathbb{E}[\text{tokens per target pass}] = \frac{1 - \alpha^{\gamma+1}}{1 - \alpha}$$

That expectation **is** the speedup factor over plain decoding (which emits exactly 1 token per
target pass), before subtracting draft-model and overhead costs. Let's tabulate it.

In [3]:
def expected_tokens(alpha, gamma):
    """E[tokens emitted per single target forward pass] under the speculative rule."""
    if alpha >= 1.0:
        return gamma + 1.0
    return (1.0 - alpha ** (gamma + 1)) / (1.0 - alpha)


alphas = [0.5, 0.7, 0.8, 0.9]
gammas = [1, 2, 4, 8]

header = "  α \\ γ  " + "".join(f"{g:>8}" for g in gammas)
print(header)
print("-" * len(header))
for a in alphas:
    row = "".join(f"{expected_tokens(a, g):>8.2f}" for g in gammas)
    print(f"  {a:.1f}    {row}")

print("\nRead: at α=0.9, γ=4 each target pass yields ~4.1 tokens — a ~4x cut in expensive passes.")
print("Note: returns DIMINISH as γ grows (α^(γ+1) → 0), and each draft token still costs draft compute,")
print("so the practical sweet spot for γ is usually 3-7, not 'as large as possible'.")

  α \ γ         1       2       4       8
-----------------------------------------
  0.5        1.50    1.75    1.94    2.00
  0.7        1.70    2.19    2.77    3.20
  0.8        1.80    2.44    3.36    4.33
  0.9        1.90    2.71    4.10    6.13

Read: at α=0.9, γ=4 each target pass yields ~4.1 tokens — a ~4x cut in expensive passes.
Note: returns DIMINISH as γ grows (α^(γ+1) → 0), and each draft token still costs draft compute,
so the practical sweet spot for γ is usually 3-7, not 'as large as possible'.


Two lessons fall out of the table:

1. **Acceptance rate dominates.** Going from α=0.5 to α=0.9 matters far more than cranking γ. Invest
   in a draft model that agrees with the target (distillation, same data, same tokenizer).
2. **γ has diminishing returns.** Each extra draft token adds less because it only pays off if *all*
   earlier ones were accepted (`α^k` shrinks fast). Past the knee, you just burn draft compute on
   tokens that get thrown away.

### 5.3 The real thing in 🤗 Transformers (optional, gated)

In practice you rarely hand-roll the loop — `model.generate(..., assistant_model=draft)` implements
**assisted generation** (HF's name for speculative decoding) for you. The code below downloads two
small checkpoints, so it is gated behind an env var and skipped by default. Set
`RUN_HF_SPECULATIVE=1` to actually run it.

In [4]:
import os

if os.getenv("RUN_HF_SPECULATIVE") and importlib.util.find_spec("transformers"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    # Target (gpt2) and a smaller draft (distilgpt2) — crucially the SAME tokenizer family.
    tok = AutoTokenizer.from_pretrained("gpt2")
    target = AutoModelForCausalLM.from_pretrained("gpt2")
    draft = AutoModelForCausalLM.from_pretrained("distilgpt2")

    prompt = "The key idea behind speculative decoding is"
    inputs = tok(prompt, return_tensors="pt")

    # Greedy is deterministic, so assisted vs. plain decoding give IDENTICAL text — only faster.
    out = target.generate(
        **inputs,
        assistant_model=draft,   # <-- this one kwarg turns on speculative decoding
        do_sample=False,
        max_new_tokens=40,
    )
    print(tok.decode(out[0], skip_special_tokens=True))
else:
    # Skipped path — still show the exact call shape so the notebook teaches without the download.
    print("Skipped (set RUN_HF_SPECULATIVE=1 and install transformers+torch to run).")
    print("Call shape:")
    print('  target.generate(**inputs, assistant_model=draft, do_sample=False, max_new_tokens=40)')
    print("\nKey point: with greedy/temperature-matched decoding the OUTPUT is identical to plain")
    print("generation — assisted generation only changes HOW MANY target forward passes it takes.")

Skipped (set RUN_HF_SPECULATIVE=1 and install transformers+torch to run).
Call shape:
  target.generate(**inputs, assistant_model=draft, do_sample=False, max_new_tokens=40)

Key point: with greedy/temperature-matched decoding the OUTPUT is identical to plain
generation — assisted generation only changes HOW MANY target forward passes it takes.


## 6. Gotchas & Pitfalls

- **Tokenizer mismatch breaks everything.** Draft and target must produce the *same token IDs*.
  Different vocabularies mean the proposed IDs don't line up with the target's logits. Use a draft
  from the same model family (or a tokenizer-aligned one).
- **A bad draft is a net *slowdown*.** If α is low, you pay for draft passes *and* near-1-token rounds.
  Always measure end-to-end latency, not just "it works." Below ~α≈0.3 the overhead can lose.
- **Batching erodes the win.** Speculative decoding reclaims *idle* memory bandwidth. At large batch
  sizes the GPU is already saturated (throughput-bound), so there's little idle slack to recover, and
  ragged accept lengths across the batch waste work. It shines most at **low batch / low latency**.
- **Sampling must match.** The losslessness proof assumes the target's sampling parameters
  (temperature, top-p, top-k) are applied consistently in the accept/resample step. Mismatched temps
  silently change the output distribution. Greedy decoding is the easy, exactly-reproducible case.
- **Don't confuse it with quantization or distillation.** Those *change* the model (lossy, smaller).
  Speculative decoding leaves the target's distribution **untouched** — it only reorganizes the
  compute. They compose: quantize the target *and* speculate.
- **γ is a tunable, not a constant.** The best draft length depends on α and on the draft/target cost
  ratio. Hard-coding γ=5 everywhere leaves speed on the table; some serving stacks adapt γ online.
- **KV-cache bookkeeping.** On rejection you must roll the target's KV cache back to the last accepted
  position. Frameworks handle this; if you hand-roll, getting it wrong corrupts later tokens.

## 7. When to Use vs Alternatives

| Approach | What it changes | Lossless? | Best when |
|---|---|---|---|
| **Speculative / assisted decoding** | Compute schedule only | ✅ Yes (same distribution) | Latency-critical, low batch, a good cheap draft exists |
| **Medusa / EAGLE / self-speculative** | Adds heads/uses own layers to draft | ✅ Yes | You want spec-decoding speedups *without* hosting a second model |
| **Prompt-lookup / n-gram drafting** | Drafts by copying from context | ✅ Yes | Summarization, code edits, RAG — lots of verbatim repetition |
| **Quantization (GPTQ/AWQ/INT8)** | Smaller weights, faster memory | ⚠️ Slightly lossy | Memory-bound *and* you can tolerate tiny quality loss; composes with spec-decode |
| **Distillation / smaller model** | Use a different, smaller model | ❌ Different model | The small model alone is good enough; you don't need the big one's quality |
| **Bigger batches / continuous batching** | Raises throughput, not per-request latency | ✅ Yes | Throughput-bound serving; *competes* with spec-decode for the same idle bandwidth |

**Rule of thumb:** speculative decoding is the go-to when you must keep the **exact** quality of a
large model but cut **single-stream latency**. If you can sacrifice a little quality, quantization is
simpler and stacks on top. If you only care about aggregate throughput, batching may matter more.
For repetitive outputs, prompt-lookup gives spec-decoding's win with *zero* extra model to host.

See also: [`kv-cache`](kv-cache.ipynb), [`quantization-gptq-awq-bnb`](quantization-gptq-awq-bnb.ipynb),
[`flash-attention`](flash-attention.ipynb), and [`vllm`](vllm.ipynb) (which ships speculative decoding).

## 8. Resources

- **Leviathan, Kalman & Matias (2023), *Fast Inference from Transformers via Speculative Decoding*** —
  the foundational paper with the acceptance-sampling proof:
  https://arxiv.org/abs/2211.17192
- **Chen et al. (2023, DeepMind), *Accelerating Large Language Model Decoding with Speculative
  Sampling*** — the parallel formulation and analysis: https://arxiv.org/abs/2302.01318
- **Hugging Face blog, *Assisted Generation*** — the practical `assistant_model` API and benchmarks:
  https://huggingface.co/blog/assisted-generation
- **Transformers docs, generation strategies (assisted / speculative decoding):**
  https://huggingface.co/docs/transformers/main/en/generation_strategies#speculative-decoding
- **Medusa — *Simple LLM Inference Acceleration with Multiple Decoding Heads*** (no separate draft
  model): https://arxiv.org/abs/2401.10774
- **EAGLE — *Speculative Sampling Requires Rethinking Feature Uncertainty*:**
  https://arxiv.org/abs/2401.15077